 created audit Delta tables to control and monitor the pipeline. The batch_control table tracks each monthly file and its processing status, such as pending, in progress, completed, or failed. The pipeline_run_log table records task-level execution details like input count, output count, status, and error messages. This makes the pipeline more reliable and easier to debug.”

In [0]:
%run ../00_setup/03-project-config

Batch table
- Which monthly file should be processed next?
- Was this batch already completed?
- Did this batch fail?
- Can I safely rerun it?
- How many rows came from this source file?

pipeline log 
- Which notebook/task ran?
- Did Bronze fail or Silver fail?
- When did the task start and end?
- How many rows went in and came out?
- What error happened?

_batch_control = status of the data file/month_

_pipeline_run_log = status of the notebook/job run_

In [0]:
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {audit_schema}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog_name}.{audit_schema}.batch_control (
    batch_id STRING,
    batch_year INT,
    batch_month INT,
    source_path STRING,
    source_file_name STRING,
    status STRING,
    start_timestamp TIMESTAMP,
    end_timestamp TIMESTAMP,
    rows_read BIGINT,
    rows_written BIGINT,
    error_message STRING,
    retry_count INT,
    created_timestamp TIMESTAMP,
    updated_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Control table for monthly batch processing status'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog_name}.{audit_schema}.pipeline_run_log (
    run_id STRING,
    task_name STRING,
    layer_name STRING,
    batch_id STRING,
    status STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    input_count BIGINT,
    output_count BIGINT,
    error_message STRING,
    created_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Pipeline execution log for monitoring Databricks tasks'
""")

print("audit control tables created successfully")

In [0]:
display(spark.sql(f"SHOW TABLES IN {catalog_name}.{audit_schema}"))

display(spark.sql(f"DESCRIBE TABLE {catalog_name}.{audit_schema}.batch_control"))

display(spark.sql(f"DESCRIBE TABLE {catalog_name}.{audit_schema}.pipeline_run_log"))